# 43008: Reinforcement Learning

## WOW Naval Simulation: MAPPO with a Fleet Commander, imitation and a league

This notebook trains a fleet of warships to win naval battles in our Unity game. Every ship is an agent (a **captain**), and a **fleet commander** gives them orders. They learn together with **MAPPO**, the multi-agent version of the PPO method from Week 10, upgraded with what the AIs that beat professional players at team strategy games did - **OpenAI Five** (Dota 2), **AlphaStar** (StarCraft II) and **Honor of Kings**: attention over units, memory, a commander/captain split, imitation of an expert first, and a league of opponents. Training follows a **curriculum** of 8 stages, from sinking a target to random fleet battles on random maps; every stage is passed at 70% of its goal.

The objective is the game's own: **get to the centre of the capture circle and hold it to score points, sink enemy ships, and do not lose your own.**

### What you will learn?
* How a Unity game becomes a multi-agent reinforcement learning environment (observation, actions, reward)
* Why the game-playing AIs use PPO + attention + memory + self-play, and how each piece maps to Week 9/10
* Define hyperparameters for MAPPO with a commander
* Imitation learning (Week 9 Part A's supervised loop) as a warm start for RL
* Performance evaluation: learning curve, pass rate per stage, Elo ratings, battle replays

**Imitation first**: run [`StageP-Imitation.ipynb`](StageP-Imitation.ipynb) once (it saves `models/imitation.pt`, which the model below starts from). The per-stage notebooks [`Stage0-Gunnery.ipynb`](Stage0-Gunnery.ipynb) … [`Stage7-Open.ipynb`](Stage7-Open.ipynb) train the same model one stage at a time.

---

### The method: MAPPO with a fleet commander

References: Yu et al., *The Surprising Effectiveness of PPO in Cooperative Multi-Agent Games* (MAPPO, NeurIPS 2022, https://arxiv.org/abs/2103.01955); OpenAI, *Dota 2 with Large Scale Deep RL* (OpenAI Five, 2019); Vinyals et al., *Grandmaster level in StarCraft II* (AlphaStar, Nature 2019); Ye et al., *Towards Playing Full MOBA Games with Deep RL* (Honor of Kings, NeurIPS 2020); Wu et al., *Hierarchical Macro Strategy Model for MOBA Game AI* (AAAI 2019); Lin et al., *TiZero* (AAMAS 2023).

Original PPO article: [Link]("https://arxiv.org/pdf/1707.06347v2.pdf")

**Why not just IPPO or MAPPO?** Every AI that beat professionals at a team strategy game used PPO-style actor-critic training with self-play, memory, attention over units and action masks - not value-decomposition methods like QMIX. So the base stays MAPPO, and the upgrades are the pieces those systems relied on:

| | PPO (Week 10) | This notebook |
|---|---|---|
| Agents | one | every ship (1 to 8) **plus a fleet commander** |
| Feature extractor | `CustomCNN` over a frame (W10-C) | **entity encoder**: every ship, contact, circle and island is a token; a transformer lets them attend to each other (AlphaStar) |
| Memory | `VecFrameStack(n_stack=4)` (W10-C) | a **GRU** per ship: what it has seen under fog of war (OpenAI Five) |
| Actions | one head | 6 heads; *which target* and *which circle* are **pointer heads** over the tokens (AlphaStar) |
| Policy | one network | **captain**: one network shared by every ship; **commander**: every 10 s an order per ship - *free*, *engage*, *hold circle k* (Honor of Kings' macro/micro split) |
| Critic | sees what the actor sees | **centralised**: the true battle, training only; one value per reward group - objective, combat, outcome - plus the chance of winning (MAPPO, Honor of Kings) |
| Update | clipped PPO | **dual-clip PPO** (Honor of Kings), value normalisation and clipping |
| Start | random | **imitation of the Elite rule AI** - Week 9 Part A's supervised loop - then RL, with a fading KL penalty towards the clone (AlphaStar) |
| Opponent | the environment | the rule AI, a passive target, or a **league** of our own past copies with PFSP and Elo (OpenAI Five, AlphaStar) |
| Reward | the agent's own | own terms blended into the fleet's (**team spirit**), minus the enemy's (**zero-sum**) (OpenAI Five) |
| An agent dies | the episode ends | **death masking**: the sunk ship stops training the actor; the critic keeps learning the team's value |

The captains and the commander only use what their team can know (fog of war included), so the trained fleet plays fair - *centralised training, decentralised execution*.

The training loop is still PPO's:

1. **Collect** `n_steps` decisions from every battle: the commander orders every 10 s, the captains act every second.
2. **Advantages** with GAE, per ship and reward group (and per order for the commander).
3. **Update** for `n_epochs` x `n_minibatches`: the critic, the captains (on chunks of 32 decisions, so the GRU replays them) and the commander.

`MAPPO(env, centralised_critic=False, commander=False)` is **IPPO**; `commander=False`, `recurrent=False`, `attn_layers=0` switch pieces off for comparisons. Read `simple_mappo/networks.py` and `simple_mappo/mappo.py` alongside this notebook.

## The environment

```
 Python (this notebook)                          Unity headless player (one battle per process)
 ┌───────────────────────┐   actions, every ship   ┌──────────────────────────────────────────┐
 │ MAPPO actor + critic  │ ──────────────────────▶ │ simulates 1 second of game time          │
 │                       │ ◀────────────────────── │ (ships, gunnery, torpedoes, fog of war)   │
 └───────────────────────┘ observation, reward     └──────────────────────────────────────────┘
                           numbers, battle over?          × n_envs battles in parallel
```

`NavalEnv` (in `simple_mappo/env.py`) runs `n_envs` battles side by side - the same idea as `make_atari_env(n_envs=4)` in the Atari notebooks. It launches the game's **headless player** (no window, no graphics) and talks to it over TCP. The enemy fleet is flown by the game's own rule-based AI, by Python as a passive target (the first stages), or - in league battles - by a frozen copy of our own fleet.

A fleet wins by reaching 1000 points (holding a circle earns points every second, each kill 12) or by sinking the whole enemy fleet. **There are no draws**: if the points are level when the clock runs out, the battle goes to overtime and the first side to pull ahead wins.

**Speed.** Training does not run at x1. Each simulated frame is a fixed `sim_dt` of game time and frames run as fast as the CPU allows. With `sim_dt = 0.08` (the physics step the game itself uses at x8 time compression) one battle runs at about **260-400x real time** - a 6-minute battle takes about 1 s - with the same battle results as the normal 0.02 step.

**Observation** - what each ship knows: a set of **tokens** (`obs["self"]`, `obs["allies"]`, `obs["contacts"]`, `obs["zones"]`, `obs["obstacles"]`, each with a mask):

| Part | What it holds |
|---|---|
| self | class, hit points, 7 damaged systems, speed, heading, position, reload timers, ammunition, every consumable (ready / active / charges), fires, floods, the match state (time left, both scores, weather, mode) |
| allies | position, distance, class, hit points, what they are doing - for each friendly ship |
| contacts | position, distance, class, aspect, in gun / torpedo range, line of sight - for each enemy ship your team has spotted |
| capture circles | position, **distance**, radius, owner, capture progress, **"I am inside"** |
| obstacles | the nearest islands, rocks and smoke clouds |

**One method for every stage**: the observation always has room for 8 ships a side and 5 circles, so the *same* network - one model, one checkpoint - trains through every stage of the curriculum (and any stage added later); a duel just leaves seven ship slots empty, and the attention masks them out. The critic gets `obs["critic_*"]`: the *true* state of both fleets and every circle. The commander gets the same with everything the team cannot know zeroed (`obs["fleet_enemy"]`, `obs["fleet_match"]`).

**Actions** - every ship picks one option from each of 6 heads (illegal options are masked out):

| Head | Options |
|---|---|
| move | keep course, 8 compass directions, close in, broadside, open range, regroup, take cover, return to port, **sail to circle k** |
| speed | full, 2/3, 1/3, stop, astern |
| target | auto, or focus enemy contact k |
| fire | fire at will, hold fire |
| torpedo | none, launch at the target |
| ability | none, one of 12 consumables (shell type, smoke, radar, repair...), dive, surface |

### The reward

Defined in `simple_mappo/rewards.py` - the one file to edit when you want the fleet to behave differently. **Ship** terms are earned by one ship for what it did; **team** terms are shared by the whole fleet. The terms fall into three **groups**, and the critic predicts each group's return separately.

| Term | Weight | Kind | Group | Paid for |
|---|---|---|---|---|
| `in_circle` | +0.05 | ship | objective | every second inside *its* circle |
| `approach_circle` | +1 | ship | objective | every km sailed towards *its* circle's centre (minus when sailing away) |
| `zone_captured` / `zone_lost` | +1 / -1 | team | objective | a circle captured / lost |
| `survive` | 0 (0.02 in Defend) | ship | objective | every second afloat |
| `enemy_sunk` / `ship_lost` | +1 / -1 | team | combat | each enemy ship sunk / each of ours |
| `damage_dealt` / `damage_taken` | +0.5 / -0.25 | ship | combat | per whole hull of damage done / taken |
| `friendly_fire` | -1 | ship | combat | damage done to our own side |
| `zero_sum` | 1 | team | combat | minus the enemy ships' average damage terms (OpenAI Five) |
| `win` / `loss` | +5 / -5 | team | outcome | the result, on the last step |

*Its* circle is the commander's: a ship ordered to hold circle k is paid for circle k, a ship ordered to engage gets no circle pay, a free ship is paid for the nearest one. **Team spirit** τ (per stage, 0.3 → 1.0) blends each ship's own terms into the fleet's average.

The circle terms are the largest steady signal on purpose: with smaller circle weights (0.02 and 0.2) and kills worth ±2, the earlier flat MAPPO fought but never found the circle in 250 battles; with these it was sailing into the circle within about 150.

### The curriculum

Defined in `simple_mappo/curriculum.py`. Training moves to the next stage once the fleet passes **70% of its qualifying battles** (against the stage's own opponent; self-play and rehearsal battles do not count). The first stages are judged on their own skill: a passive target never scores, so the game's tiebreak would hand random play the "win".

| Stage | Battle | Fleet | Opponent | Passed when |
|---|---|---|---|---|
| P - Imitation | Elite AI vs Elite AI | - | - | (supervised: copy the Elite AI, `StageP-Imitation.ipynb`) |
| 0 - Gunnery | `gunnery`, 7 km | 1 BB vs 1 CA | passive target | target sunk within 150 s (random 52%, script 76%) |
| 1 - Capture | `capture`, one neutral circle | 1 CA | passive, far away | circle captured (random 0%, script 100%) |
| 2 - Defend | `defend`, neutral circle | 1 CA vs 1 CA | Recruit | won (random 12%) |
| 3 - Duel | `bb_duel` | 1v1 BB | Recruit | won |
| 4 - King of the Hill | `koth_3v3`, one circle | 3v3, **commander**, τ 0.3 | Veteran | won |
| 5 - Archipelago | `archipelago_3v3`, 3 circles | 3v3, commander, τ 0.5 | Veteran | won |
| 6 - Domination | procedural 6v6 | commander, τ 0.8 | **league**: Elite 40% / latest self 40% / past 20% | won vs Elite |
| 7 - Open | procedural 4-8 a side | commander, τ 1.0 | league | **final**: won vs Elite on 250 held-out maps |

* **One model for every stage**: promotion only changes the battles; the same networks keep training.
* **Rehearsal**: from stage 3 on, 10-15% of battles replay an earlier stage so its skills stay.
* **The league** (stages 6-7): a frozen copy of the current fleet, and older copies chosen by prioritised fictitious self-play (the ones we lose to come up most), with Elo ratings for everyone.
* **Held-out evaluation** (stage 7): every 100,000 timesteps the fleet plays 50 battles against the Elite AI on generated maps whose seeds training never uses; training stops once the last 250 of those reach 70%.
* **More stages**: add dicts to the list - there is no limit.
* **One stage at a time**: `Stage0-Gunnery.ipynb` ... `Stage7-Open.ipynb` train this same model one stage per notebook.

---
## Setup

### Running on Amazon SageMaker

1. On your computer, in the repository: build the headless player once (`Training/build_player.sh`), then run `Training/package_sagemaker.sh`. It writes `wow_sagemaker.zip` (code + scenarios + the player, about 40 MB).
2. Upload `wow_sagemaker.zip` to SageMaker Studio (drag it into the file browser) and run the cell below.
3. Pick an instance with **many CPUs** for the JupyterLab space (e.g. `ml.c5.4xlarge`, 16 vCPU): every parallel battle is one Unity process on its own core. No GPU is needed; the networks are small.
4. The whole curriculum takes millions of timesteps - for that, use the **training job** at the end of this notebook, so it does not depend on the notebook staying open.

On your own computer, skip this cell and open the notebook from the repository's `Training/` folder.

In [ ]:
import os, time, zipfile
# SageMaker Studio (/home/sagemaker-user) or a SageMaker notebook instance (/home/ec2-user/SageMaker)
ON_SAGEMAKER = any(os.path.isdir(p) for p in ("/home/sagemaker-user", "/home/ec2-user/SageMaker"))
home = os.path.expanduser("~/SageMaker") if os.path.isdir(os.path.expanduser("~/SageMaker")) else os.path.expanduser("~")
bundle = os.path.join(home, "wow_sagemaker.zip")
if ON_SAGEMAKER and os.path.basename(os.getcwd()) != "Training":
    if not os.path.isdir(os.path.join(home, "wow")):
        assert os.path.exists(bundle), f"upload wow_sagemaker.zip to {home} first"
        zipfile.ZipFile(bundle).extractall(os.path.join(home, "wow"))
    %cd {home}/wow/Training

# Uploading a new wow_sagemaker.zip does not replace the unzipped folder: check which version runs here
running = open("BUNDLE_VERSION").read().strip() if os.path.exists("BUNDLE_VERSION") else "an older one"
if ON_SAGEMAKER and os.path.exists(bundle):
    with zipfile.ZipFile(bundle) as z:
        uploaded = z.read("Training/BUNDLE_VERSION").decode().strip() if "Training/BUNDLE_VERSION" in z.namelist() else "unknown"
    if uploaded != running:
        print(f"*** {bundle} is version {uploaded}, but this notebook runs version {running}. To switch, run in a terminal:\n"
              f"      cd {home} && mv wow wow_old_{time.strftime('%Y%m%d_%H%M')} && "
              f"python -c \"import zipfile; zipfile.ZipFile('wow_sagemaker.zip').extractall('wow')\"\n"
              f"    then close this tab and open wow/Training/WOW-MAPPO-Simple.ipynb again. ***")
print(os.getcwd(), "| bundle", running, "| glibc", os.confstr("CS_GNU_LIBC_VERSION"))   # the Unity player needs glibc 2.35+

## Install Packages and Dependencies

In [ ]:
# PyTorch, NumPy, pandas, matplotlib and Pillow are already on SageMaker images and Anaconda; install only
# what is missing (PyTorch as the CPU build - training runs on the CPU and the GPU build is ~2.5 GB)
import importlib.util
missing = [pkg for mod, pkg in (("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"))
           if importlib.util.find_spec(mod) is None]
if missing:
    !pip install -q {" ".join(missing)}
if importlib.util.find_spec("torch") is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu

## Importing required libraries and modules

In [ ]:
import os
import json
import copy
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from simple_mappo import (NavalEnv, MAPPO, Monitor, BaseCallback, CurriculumCallback, BestModelCallback,
                          DEFAULT_CURRICULUM, REWARD_WEIGHTS, TeamStorage, TeamSyncCallback, VideoCallback,
                          training_status, evaluate, plot_results, plot_curriculum, plot_reward_terms, plot_progress,
                          record_battle, show_videos, view, find_unity_binary)

print(f"{torch.__version__=}  cuda={torch.cuda.is_available()}  CPUs={os.cpu_count()}")
print("headless player:", find_unity_binary())

## Team checkpoints

The whole team trains **one shared line of training** in turns. Its checkpoints and logs live in one S3 folder,
`<TEAM_STORAGE>/runs/<RUN_NAME>/`. When you start, the notebook pulls the latest checkpoint from there and continues it (curriculum stage included); while you train it pushes a new checkpoint every `save_interval` timesteps, and once more at the end. The learning curve in `logs/monitor.csv` keeps growing across everybody's sessions.

* Everyone must use the **same `TEAM_STORAGE` and `RUN_NAME`** to continue the same training. On SageMaker the default is your account's SageMaker bucket - if teammates log in to *different* AWS accounts, create one bucket, give everyone access, and put its name here.
* Only one person can train a run at a time: if a teammate is training it, the notebook says so and stops. Use another `RUN_NAME` to train something different in parallel (e.g. a reward experiment).
* `TEAM_STORAGE = None` trains on this machine only.
* While anyone trains, the run's **status** (stage, progress, speed, time left, recent win rate, newest video) is updated in the folder every few minutes - see *Training status* below.

In [ ]:
YOUR_NAME = "your-name"              # shown to your teammates in the shared run
RUN_NAME = "fleet"                   # the line of training the team takes turns extending (as in the stage notebooks)
if ON_SAGEMAKER:
    from simple_mappo.aws import team_storage_uri       # boto3 only: works with any SageMaker SDK version
    TEAM_STORAGE = team_storage_uri()                    # s3://sagemaker-<region>-<account>/wow-mappo, or "s3://<team-bucket>/wow-mappo"
else:
    TEAM_STORAGE = None              # e.g. "s3://<your-team-bucket>/wow-mappo" (needs boto3), or a shared folder

storage = TeamStorage(TEAM_STORAGE, RUN_NAME, user=YOUR_NAME) if TEAM_STORAGE else None
if storage:
    print("team run:", storage.location)
    print("last push:", storage.info() or "none yet - you will start it")

## Create Directories for Model and Logs Storage

In [ ]:
# Change the paths based on your directory structure
run_dir = f"runs_simple/{RUN_NAME}"
log_dir = os.path.join(run_dir, "logs/")
models_dir = os.path.join(run_dir, "models/")
video_dir = os.path.join(run_dir, "videos/")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(video_dir, exist_ok=True)

## Callback for Model Checkpointing

The same callback as Weeks 9 and 10. One difference: with `n_envs` parallel battles `num_timesteps` grows by `n_envs` per step, so it seldom lands exactly on a multiple of `save_interval` - this version saves whenever it *crosses* one.

In [ ]:
class SaveOnIntervalCallback(BaseCallback):
    def __init__(self, save_interval: int, save_path: str, verbose=1):
        super().__init__(verbose)
        self.save_interval = save_interval
        self.save_path = save_path
        self.saved = 0

    def _on_training_start(self) -> None:
        self.saved = self.num_timesteps // self.save_interval
        if self.num_timesteps == 0:
            self.model.save(os.path.join(self.save_path, "model_0"))   # the untrained policy, for the "beginning" replay

    def _on_step(self) -> bool:
        if self.num_timesteps // self.save_interval > self.saved:
            self.saved = self.num_timesteps // self.save_interval
            save_file = os.path.join(self.save_path, f'model_{self.num_timesteps}')
            self.model.save(save_file)
            if self.verbose > 0:
                print(f'Saving model to {save_file}.pt')
        return True

## Configuration and Environment Initialization

A **timestep** is one decision of one battle (every ship decides once; one decision = `decision_period` seconds of game time). With `n_envs = 8`, 8 timesteps are collected per step. 300,000 timesteps take about 30-40 minutes on 8 CPU cores - enough to see stage 0 learning; the whole curriculum needs millions (use the training job).

In [ ]:
# Define hyper parameters for MAPPO with a fleet commander
config = OrderedDict([
    # the battles
    ('curriculum', copy.deepcopy(DEFAULT_CURRICULUM)),   # the stages above (edit, or add your own)
    ('start_stage', 0),
    ('n_envs', max(1, min(16, os.cpu_count() // 2))),     # battles in parallel, one Unity process each
    ('decision_period', 1.0),          # game seconds between decisions
    ('sim_dt', 0.08),                  # game seconds per frame: 0.02 normal, 0.08 fastest stable
    # PPO (Week 10)
    ('n_steps', 256),                  # decisions per battle before each update
    ('n_minibatches', 4),              # each update splits the collected data into 4 minibatches...
    ('n_epochs', 5),                   # ...5 times: 20 gradient steps, the same for a 1v1 duel or an 8v8 battle
    ('gamma', 0.995),                  # battles are long: 0.995 looks about 200 s ahead
    ('gae_lambda', 0.95),
    ('clip_range', 0.2),
    ('dual_clip', 3.0),                # Honor of Kings: bounds the loss of a negative advantage
    ('ent_coef', 0.01),                # entropy bonus, decaying over training (or "auto", as in SAC)
    ('learning_rate', 3e-4),           # captains
    ('critic_learning_rate', 5e-4),
    # the networks
    ('d_model', 64),                   # token size of the entity encoder
    ('attn_layers', 2),                # transformer layers (0 = pooling only)
    ('hidden_size', 128),              # each ship's GRU memory
    ('recurrent', True),
    ('commander', True),               # the fleet commander (orders in stages with "commander": True)
    ('commander_period', 10),          # decisions between its orders
    ('centralised_critic', True),      # True = MAPPO; False = IPPO (needs commander=False), to compare
    # imitation and league
    ('kl_bc_coef', 0.1),               # KL penalty towards the imitation policy, fading out
    ('opponent_refresh', 10),          # league: updates between refreshing the frozen latest copy
    ('n_timesteps', 300_000),          # Adjust this depending on your training limit
])
pd.DataFrame(config['curriculum']).fillna("")

In [ ]:
# The reward weights (see the table above). Change one here, e.g. reward_weights['in_circle'] = 0.1
reward_weights = OrderedDict(REWARD_WEIGHTS)
pd.Series(reward_weights, name="weight").to_frame()

In [ ]:
def make_env():
    # launches n_envs headless Unity players; the observation is sized for the biggest stage
    env = NavalEnv(stages=config['curriculum'], start_stage=config['start_stage'], n_envs=config['n_envs'],
                   decision_period=config['decision_period'], sim_dt=config['sim_dt'],
                   reward_weights=reward_weights, unity_log_dir=os.path.join(run_dir, "unity_logs"))
    return Monitor(env, log_dir)

env = make_env()
print("battles in parallel:", env.n_envs, " ship slots per side:", env.n_agents)
print("token sizes:", env.dims)
print("action heads:", dict(zip(env.head_names, env.head_sizes)))
print("current stage:", env.stages[env.stage]["name"])

### Look inside one observation

Every number has a name - Unity sends the feature names with the environment's description. Here are the first ship's capture-circle features, which the `in_circle` and `approach_circle` rewards are computed from.

In [ ]:
obs = env.reset()
print({k: v.shape for k, v in obs.items()})

raw = env.raw_arrays(0)                         # everything Unity sent for battle 0, both fleets
zone_names = env.spec.features["zone"]
circle = raw["zones"][0, 0, 0]                  # [fleet 0, ship 0, circle 0]
pd.Series(circle, index=zone_names, name="ship 0, circle 0").to_frame().T

## Baselines

Before training, measure what a policy has to beat on the first stage:
* **Random**: a random legal option on every head.
* **Scripted**: every ship sails to the circle at full speed and fires at will (a sensible hand-written tactic).

In [ ]:
class GoToCircle:
    # move head: 15 fixed options, then one per circle; speed 0 = full; target 0 = auto; fire 0 = at will
    def predict(self, obs, deterministic=False):
        E, N = obs["alive"].shape
        actions = np.zeros((E, N, len(env.head_sizes)), dtype=np.int32)
        actions[..., 0] = np.where(obs["action_mask"][..., 15] > 0.5, 15, 0)
        return actions

baselines = {"random": evaluate(None, env, n_battles=24),
             "scripted: go to circle": evaluate(GoToCircle(), env, n_battles=24)}
pd.DataFrame(baselines).T[["stage", "win_rate", "pass_rate", "win_rate_stderr", "mean_reward"]]

(The baseline battles are logged as evaluation battles; the learning-curve plots leave them out.)

## Model Initialization and Training

In [ ]:
# Continue the team's training if the shared run already has a checkpoint, otherwise start a new model
latest = storage.pull(run_dir) if storage else None
if latest:
    model = MAPPO.load(latest, env=env, log_dir=log_dir, verbose=1)
    print(f"continuing at {model.num_timesteps:,} timesteps on stage '{env.stages[env.stage]['name']}'")
else:
    model = None
if storage:
    storage.lock()                   # stops here if a teammate is training this run right now

In [ ]:
# Create the model (skipped when continuing the team's run)
if latest is None:
    model = MAPPO(
        env,
        learning_rate=config['learning_rate'],
        critic_learning_rate=config['critic_learning_rate'],
        n_steps=config['n_steps'],
        n_minibatches=config['n_minibatches'],
        n_epochs=config['n_epochs'],
        gamma=config['gamma'],
        gae_lambda=config['gae_lambda'],
        clip_range=config['clip_range'],
        dual_clip=config['dual_clip'],
        ent_coef=config['ent_coef'],
        d_model=config['d_model'],
        attn_layers=config['attn_layers'],
        hidden_size=config['hidden_size'],
        recurrent=config['recurrent'],
        commander=config['commander'],
        commander_period=config['commander_period'],
        centralised_critic=config['centralised_critic'],
        kl_bc_coef=config['kl_bc_coef'],
        opponent_refresh=config['opponent_refresh'],
        verbose=1,
        log_dir=log_dir,
    )
    # start the captains from the clone of the Elite AI, if StageP-Imitation.ipynb made one
    imitation = os.path.join(models_dir, "imitation.pt")
    if os.path.exists(imitation):
        model.init_from_imitation(imitation)

save_interval = config['n_timesteps'] // 3
callback = [SaveOnIntervalCallback(save_interval, save_path=models_dir),
            CurriculumCallback(),          # promotes to the next stage, and applies the final stage's stopping rule
            BestModelCallback(models_dir), # keeps models/model_best.pt: the best rolling pass rate
            VideoCallback(every=config['n_timesteps'] // 6, video_dir=video_dir)]   # a replay video now and then
if storage:   # share progress with the team: a status update every 10,000 timesteps, a checkpoint every save_interval
    callback.append(TeamSyncCallback(storage, run_dir, push_every=save_interval, status_every=10_000))

print(model.actor)       # the captain and the commander
print(model.critic)

`n_timesteps` is how many timesteps to train *in this session* (continuing a team run adds to what is already there).

Each printed table is one update. What to watch:

| Line | Meaning | Healthy |
|---|---|---|
| `stage` | the curriculum stage new battles start on | goes up |
| `ep_rew_mean` | average reward per ship per battle | goes up |
| `win_rate` / `pass_rate` | share of battles won / passed (the stage's goal) against its own opponent since the last table | goes up |
| `win_rate_self_play` | the same against our own copies (stages 6-7) | about 0.5 |
| `entropy` | how random the captains still are (6 heads summed) | falls slowly, not to 0 |
| `approx_kl` | how far one update moved the policy | about 0.005 - 0.03 |
| `clip_fraction` | share of samples PPO's clip stopped | below about 0.3 |
| `explained_variance` | how well the critic predicts the returns | rises towards 1 |
| `orders_free` | share of the commander's orders that leave the ship free (stages 4+) | falls as it learns to command |
| `kl_bc` | how far the captains moved from the imitation policy | rises slowly |
| `elo` | the fleet's Elo rating against the rule AI and its past copies | goes up |

In [ ]:
# Start the training (a progress table is printed every log_interval updates)
model.learn(total_timesteps=model.num_timesteps + config['n_timesteps'], callback=callback, log_interval=5)

## Save Model

Saves the final model and hands the run back to the team. **If you stop training early (Kernel > Interrupt), run this cell too**, so your progress is pushed and the lock released.

In [ ]:
# Save the final model (it remembers the curriculum stage it reached)
model.save(os.path.join(models_dir, "model_final"))
model.save(os.path.join(models_dir, "model_latest"))
if storage:
    storage.push(run_dir, model, status=training_status(model, run_dir))   # checkpoint, logs, videos, status
    storage.release()                # a teammate can continue from here

## Training status

How the team's run is doing - also while a teammate or a training job is training it. `show_status()` reads the run's status in the shared folder; `pull_view()` downloads its logs and newest replay videos (without disturbing whoever is training) for the plots below.

In [ ]:
if storage:
    storage.show_status()
    view_dir = storage.pull_view(os.path.join(run_dir, "team_view"), videos=3)
    show_videos(os.path.join(view_dir, "videos"))          # the newest replays recorded during training

## Plot the Learning Curve

In [ ]:
plot_results(log_dir)

Pass rate against each stage's own opponent, one colour per curriculum stage - a new colour is a promotion:

In [ ]:
plot_curriculum(log_dir)

Which reward terms the fleet is earning, battle by battle. A rising `in_circle` means the ships learned to hold the circle; `ship_lost` moving towards 0 means they stopped dying.

In [ ]:
plot_reward_terms(log_dir)

In [ ]:
plot_progress(log_dir)

## Performance Evaluation

Fresh battles against the rule AI on every stage reached so far, with sampled actions (a young policy is often close to 50/50 on choices like fire / hold fire, and always taking the likelier option can mean never firing).

In [ ]:
results = {f"{name} ({r['stage']})": r for name, r in baselines.items()}
for s in range(env.stage + 1):
    results[f"MAPPO on {env.stages[s]['name']}"] = evaluate(model, env, n_battles=24, stage=s)
pd.DataFrame(results).T[["battles", "win_rate", "pass_rate", "win_rate_stderr", "mean_reward", "enemy_ships_sunk", "own_ships_lost"]]

## Record and Display Replays

Everything here is the real Unity game - the headless player is a build of the project, with the same ships, gunnery, detection, physics and AI - but it runs **without graphics** (a server has no screen), so these replays are a **2D map drawn from the game's data**: positions, headings, hit points, capture circles. To see a trained model in the game's own graphics, use `watch_simple.py` on a computer with a screen (see *Next steps*).

Replays were recorded during training (in `videos/`, and in the team folder). Below: one battle each at the beginning, middle and end of training, each on the stage that model had reached - MP4 where ffmpeg is installed, GIF otherwise. Cyan is our fleet, red the enemy, the dashed ring a capture circle (its colour shows the owner), and the green bar each ship's hit points.

In [ ]:
view(models_dir, env, video_folder=video_dir)

In [ ]:
# Display the replays created at different stages of the trained model
for stage in ["beginning", "middle", "end"]:
    show_videos(video_dir, prefix=f'mappo-naval-{stage}')

In [ ]:
env.close()

---
## Next steps

* **Keep training**: just run the notebook again - with team checkpoints it pulls the newest one (yours or a teammate's) and continues. Without them: `model = MAPPO.load(os.path.join(models_dir, "model_final"), env=env)` then `model.learn(...)`.
* **Add a stage** (there is no limit): append a dict to `config['curriculum']`, e.g. a Capture and Control stage:
  `{"name": "8 Capture and Control", "procedural": {"mode": 3, "preset": "random", "ships": [6, 8]}, "difficulty": 2, "opponent": "league", "opponents": {"rule": 0.4, "latest": 0.4, "past": 0.2}, "commander": True, "team_spirit": 1.0, "promote_win_rate": 0.7, "window": 200}`.
  Move the `stop_win_rate` rule to whichever stage is last. The same model carries on - any stage up to 8 ships a side and 5 circles fits the network.
* **Change the behaviour**: edit the weights in `simple_mappo/rewards.py` (or `reward_weights` above) and compare the learning curves.

### See a trained model in the real game (on your own computer)

SageMaker has no screen, so Unity cannot draw there. Download a model (Studio file browser: right-click `models/model_final.pt` > Download), then in the repository's `Training/` folder:

```bash
# a video rendered by Unity itself (fog of war lifted so both fleets show)
python watch_simple.py --model model_final.pt                 # -> runs_simple/watch/model_final_stage<k>_1.mp4

# or play it live in the Unity editor: tick GameBootstrap > Rl Training Server, press Play, then
python watch_simple.py --model model_final.pt --live
```

`--stage 3` picks another curriculum stage; by default it plays the stage the model reached.

---
## (Optional) Train the whole curriculum as a SageMaker Training Job

Runs `train_simple.py` on a separate machine, so training does not depend on this notebook staying open or the space's idle shutdown. It uses AWS's PyTorch CPU container (Ubuntu 22.04, which the Unity player needs) with the headless player as its input. The job trains the **same team run**: it pulls the latest checkpoint, pushes status every 10,000 timesteps and a checkpoint and a replay video every `save-interval` - so you watch it with *Training status* above, and a teammate can continue its training in a notebook afterwards. (A job and a notebook cannot train the same `RUN_NAME` at once: the second one is refused.)

Built on boto3, so it works with any SageMaker SDK version. **Course accounts often do not allow training jobs** (an *AccessDenied ... explicit deny* error) or have no quota for the instance type - the cell then says so. Train in the notebook instead: the team folder keeps every checkpoint, so each session - yours or a teammate's - continues where the last one stopped.

In [ ]:
from simple_mappo.aws import launch_training_job, job_status, stop_training_job, list_training_jobs

job = launch_training_job(
    RUN_NAME, TEAM_STORAGE, user=f"{YOUR_NAME}-job",
    hyperparameters={"total-timesteps": 20_000_000,     # the run's total; the job stops there
                     "n-envs": 8,                        # battles in parallel (one per 2 vCPU)
                     "save-interval": 100_000},          # checkpoint + replay video this often
    instance_type="ml.c5.4xlarge",                       # 16 vCPU, 32 GB, no GPU needed
    max_hours=48)                                        # it continues from the team run if you start another

**Job status**: AWS's view of the job (starting, training, failed and why) and the newest lines of its log, then the training's own progress from the team folder. Run this cell whenever you want an update; the job's page in the SageMaker console (link printed above) also charts `win_rate`, `ep_rew_mean` and the other table values under *Monitor*.

In [ ]:
job_status(job)
print()
storage.show_status()

In [ ]:
# stop_training_job(job)     # stops it; the team keeps its last pushed checkpoint
# list_training_jobs()       # your recent jobs, e.g. to find a job name after restarting the notebook